In [36]:
from feat_engin_helpers import *
import data_loaders

data = data_loaders.DataLoader()
Y, cases, unit_ids = load_uc()
long = to_long(Y, cases, unit_ids)

features = []

#2. Attach raw features
price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"]
attach_hourly(long, price, "price")
features.append("price")

inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv")
inflow = inflow.loc['2015-01-01':]
for res in inflow:
    attach_hourly(long, inflow[res], "inflow_"+res)
    features.append("inflow_"+res)

storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv")
for res in storage:
    if res == "Hyljelihyl":
        continue
    attach_daily(long, storage[res], "storage"+res)
    features.append("storage"+res)

value = read_timeseries("../sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv")
for res in value:
    attach_daily(long, value[res], "value_"+res)
    features.append("value_"+res)

min_vol = read_timeseries("../sintef_project/data/extended/Constraint_min_volume.csv")
for res in min_vol:
    attach_hourly(long, min_vol[res], "minVolume_"+res)
    features.append("minVolume_"+res)

min_flow = read_timeseries("../sintef_project/data/extended/Constraint_min_flow.csv")
for res in min_flow:
    attach_hourly(long, min_flow[res], "minFlow_"+res)
    features.append("minFlow_"+res)

# 3. New features
attach_case_level(long, cases, value["Kjelavatn"], "wv_kjela_start", offset_days=0)
attach_case_level(long, cases, value["Kjelavatn"], "wv_kjela_end",   offset_days=7)

long["price_minus_wv_end"] = long["price"] - long["wv_kjela_end"]
long["price_rank_in_case"] = long.groupby("case_idx")["price"].rank(pct=True)
features.append("price_minus_wv_end")
features.append("price_rank_in_case")

long["rolling_prices_day"] = long["price"].rolling(window=24,min_periods=1).sum()
features.append("rolling_prices_day")
long["rolling_prices_week"] = long["price"].rolling(window=24*7,min_periods=1).sum()
features.append("rolling_prices_week")

long["day_of_week"] = long["timestamp"].dt.dayofweek
long["month"] = long["timestamp"].dt.month
features.append("hour_of_day")
features.append("day_of_week")
features.append("month")

In [37]:
## TO DO!!!!!!!!!!!! kies trainingsset om horizon aan te passen
sub = long[long["case_idx"] % 7 == 0]
train = sub[sub["year"] <= 2021]

for unit, g in train.groupby("unit", observed=True):
    scores = filter_scores(g, features, "y")
print(scores)

  feature  correlation       auc  strength  sign_consistency
0   price      0.01605  0.509269  0.009269          0.428571
